# Notebook 15 — Final Benchmark: Baselines, Correctness Tests and the Combined Table

**Hyperparameter-optimisation experiment · Soccer Event Transformer · Track A model family**

| Item | Content |
|---|---|
| **Question** | Does the selected Transformer improve on the baselines when both are trained and evaluated under the same final setup, and do the implementation's correctness tests hold at the selected configuration? |
| **Why the baselines are re-run** | The existing baseline results were produced under the Track A training setup (weight decay 0.1; the MLP baseline B4 with development seeds 0–2). The selected model was trained after the hyperparameter search (weight decay 0.01; final seeds 100–104). A fair comparison needs both under one setup. |
| **Purpose of this notebook** | (1) Re-run baselines B1a, B1b, B2, B3, B4 under the final conditions. (2) Execute the 16 correctness tests at the selected configuration. (3) Apply the pre-registered validation gates to the selected model against the re-run baselines. (4) Evaluate the re-run baselines once on test. (5) Build one benchmark table from the re-run baselines and the recorded Transformer results. |
| **What is not done here** | The selected Transformer is not retrained and not re-evaluated on test; its results are read from its saved files. No hyperparameter of any model is tuned. |
| **Test data** | Used once, for the re-run baselines only, after their plan and validation results are written (Section 8). |
| **Outputs** | Under `experiments/hyperparameter_optimisation/final_benchmark/`: `baseline_final_benchmark.csv`, `combined_final_benchmark.csv`, `correctness_tests_final_configuration.csv`, `15_*` result files, plan, manifest and summary. |



## 0. Experimental provenance

### 0.1 Models in this notebook

| Label | Model | Training setup | Role here |
|---|---|---|---|
| **B1a, B1b, B2, B3** | non-neural baselines: class prior, zone prior, Markov (previous action × zone), logistic regression on the current event | fitted on the training split; no training setting | re-run here |
| **B4** | MLP over the current and three previous events (512 → 512 → 512 → 15) | learning rate 9e-4, weight decay 0.01, seeds 100–104 | re-trained here |
| **SEL-4L** | the selected model: 4 layers, 2 heads, d_model 128, d_ff 512, K = 16, T300 | learning rate 3e-4, weight decay 0.01, seeds 100–104; primary checkpoint seed 103 | results read from file |
| **REF-2L** | the frozen Track A model: 2 layers, otherwise as above | Track A recipe: weight decay 0.1, seeds 100–104 | results read from file |
| **REF-4L** | the 4-layer model of a separate, earlier depth check | Track A recipe: weight decay 0.1, seeds 100–104 | results read from file |

REF-2L and REF-4L are reference Transformer configurations from model development. REF-4L is not in the lineage of the selected model; it appears only as a labelled row of the combined table and in one paired comparison with SEL-4L (same architecture, different weight decay).

### 0.2 Where each component comes from

| Component | Source | Status here |
|---|---|---|
| Data, T300 encoding, pre-action possession features, splits, windows, evaluation views | `artifacts/` | read only |
| Baseline definitions B1a, B1b, B2, B3 and their refit helper | `src/baselines.py`, `src/evaluation.py` | reused unchanged |
| B1b and B2 smoothing strength α | selected on validation when the baselines were first fitted (α = 10 for both); `results/baselines_validation.json` | reused, not re-tuned |
| B3 settings (C = 1.0, up to 1,000 L-BFGS iterations) | fixed before the first fit | reused, not tuned |
| B4 definition | `src/mlp_baseline.py` | reused unchanged |
| B4 learning rate (9e-4) | selected on validation for the MLP; `results/04_b4_comparison.json` | reused, not re-tuned |
| Model code, training loop, metrics, bootstrap, decision rule | `src/` | imported, never rewritten |
| Correctness tests | `tests/test_model.py` | re-executed at the selected configuration |
| Selected configuration | `experiments/hyperparameter_optimisation/config/frozen_config.json` | read only; hash recorded |
| SEL-4L runs and test predictions | `13_final_runs.csv`, `results/test_predictions/` of the optimisation experiment | read only |
| Recorded Transformer test results | `13_test_results.json` (SEL-4L, REF-2L); `benchmarks/track_a_4layer/results/06_4L_test_results.json` (REF-4L) | loaded programmatically |

### 0.3 What changes relative to the original baseline benchmark

| Baseline | Setting that depends on the training setup | Original baseline benchmark | Final benchmark (this notebook) | Reason |
|---|---|---|---|---|
| B1a, B1b, B2 | none (count tables; α fixed) | refit on the training split | refit from scratch, same α | nothing changes; the refit checks reproducibility |
| B3 | none (C and iteration budget fixed) | loaded from a saved fit | fitted from scratch | same fit expected; the refit checks reproducibility |
| B4 | weight decay | 0.1, the Track A Transformer setting | **0.01**, the final Transformer setting | B4 uses the weight decay selected for the Transformer, with its own learning rate |
| B4 | seeds | development seeds 0–2 | **final seeds 100–104** | the final benchmark reports seeds that played no part in any decision |
| B4 | epoch budget, early stopping, warm-up, batch size | Track A values | final setting (identical values) | unchanged |

### 0.4 What remains identical

Dataset, preprocessing, event filtering, target classes, splits, identity vocabularies, windows (K = 16, stride 8), the one-prediction-per-event evaluation view, metric definitions (NLL, accuracy, top-3, top-5, macro-F1, ECE with 15 bins), the match-level bootstrap (2,000 resamples), the decision rule, and the B4 learning rate and B1b/B2 α.

### 0.5 Procedural inconsistencies found and how they are handled

* **B4 seeds.** In the original baseline benchmark B4 used development seeds 0–2, while the restricted Transformers used 100–102 and the final Transformer 100–104. This is an inconsistency rather than a conflict between two procedures; it is resolved here by applying the final seed policy (100–104) to B4.
* **B4 in the earlier 4-layer depth check.** There the Track A B4 runs were reused because the weight decay equalled Track A's. That condition does not hold for the selected model (0.01 against 0.1), so B4 is retrained here.


## 1. Environment

In [ ]:
# ---------------------------------------------------------------------------
# Environment setup (same layout as notebooks 08-14 of this experiment)
# ---------------------------------------------------------------------------
import os
import sys
import json
import time
import shutil
import hashlib
import importlib
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT_DIR = Path(os.environ.get("TRACK_A_DIR", "/content/drive/MyDrive/soccer_mi/track_a"))
RUN_MODE = os.environ.get("TRACK_A_RUN_MODE", "full")
assert RUN_MODE in ("full", "smoke")
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

EXISTING_DIR = PROJECT_DIR if RUN_MODE == "full" else PROJECT_DIR / "smoke"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
EXISTING_RUNS_DIR = EXISTING_DIR / "runs"
EXISTING_RESULTS_DIR = EXISTING_DIR / "results"
EXISTING_FROZEN_CONFIG = EXISTING_DIR / "config" / "frozen_config.json"
BENCH_DIR = EXISTING_DIR / "benchmarks" / "track_a_4layer"
OPT_DIR = EXISTING_DIR / "experiments" / "hyperparameter_optimisation"
OPT_RESULTS_DIR = OPT_DIR / "results"
OPT_CONFIG_PATH = OPT_DIR / "config" / "frozen_config.json"

WORK_DIR = OPT_DIR / "final_benchmark"
RUNS_DIR = WORK_DIR / "runs"
RESULTS_DIR = WORK_DIR / "results"
FIGURES_DIR = WORK_DIR / "figures"
CONFIG_DIR = WORK_DIR / "config"
MANIFEST_DIR = WORK_DIR / "manifests"
LOG_DIR = WORK_DIR / "logs"
for folder in [RUNS_DIR, RESULTS_DIR, FIGURES_DIR, CONFIG_DIR, MANIFEST_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = MANIFEST_DIR / "experiment_manifest.jsonl"


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_json(path):
    with open(path) as handle:
        return json.load(handle)


print("Colab:", IN_COLAB, "| RUN_MODE:", RUN_MODE)
print("Outputs of this notebook:", WORK_DIR)

Mounted at /content/drive
Colab: True | RUN_MODE: full
Outputs of this notebook: /content/drive/MyDrive/soccer_mi/track_a/experiments/hyperparameter_optimisation/final_benchmark


In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt


import src.data as data
import src.batching as batching
import src.metrics as metrics
import src.model as model_module
import src.training as training
import src.analysis as analysis
import src.evaluation as evaluation
for module in [data, batching, metrics, model_module, training, analysis, evaluation]:
    importlib.reload(module)

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if RUN_MODE == "full" and DEVICE != "cuda":
    raise RuntimeError("BLOCKED: full-mode training requires a GPU. In Colab: Runtime > Change runtime type > GPU.")
N_BOOT = 2000 if RUN_MODE == "full" else 50

events = pd.read_parquet(ARTIFACTS_DIR / "events_prepared.parquet")
targets = events["action_id"].to_numpy().astype("int64")
match_ids = events["match_id"].to_numpy()

TRACK_A_FROZEN = load_json(EXISTING_FROZEN_CONFIG)
TRACK_A_HYPERPARAMETERS = load_json(EXISTING_RESULTS_DIR / "04_selected_hyperparameters.json")
SELECTED = load_json(OPT_CONFIG_PATH)

ENCODING = TRACK_A_FROZEN["encoding"]
POSSESSION_FEATURES = TRACK_A_FROZEN["possession_features"]
STRIDE = TRACK_A_FROZEN["stride"]
FINAL_CONTEXT = SELECTED["context_length"]
FINAL_SEEDS = SELECTED["seed_policy"]["final_seeds"]

print("Events:", len(events), "| device:", DEVICE)
print("Selected configuration (notebook 12):",
      json.dumps({key: SELECTED[key] for key in ["context_length", "model", "training"]}))
print("Final seeds:", FINAL_SEEDS)

Events: 658905 | device: cuda
Selected configuration (notebook 12): {"context_length": 16, "model": {"n_layers": 4, "n_heads": 2, "d_model": 128, "d_ff": 512}, "training": {"learning_rate": 0.0003, "weight_decay": 0.01, "max_epochs": 30, "batch_size": 256, "warmup_fraction": 0.05, "min_lr_ratio": 0.1, "grad_clip": 1.0, "patience": 5}}
Final seeds: [100, 101, 102, 103, 104]


### 1.1 Snapshot of every existing result file



In [ ]:
def fingerprint_folder(folder, patterns=("*.json", "*.csv", "*.npz", "*.jsonl")):
    fingerprints = {}
    for pattern in patterns:
        for path in sorted(Path(folder).rglob(pattern)):
            if WORK_DIR in path.parents:
                continue
            fingerprints[str(path.relative_to(EXISTING_DIR))] = file_sha256(path)
    return fingerprints


SNAPSHOT_FOLDERS = [EXISTING_RESULTS_DIR, BENCH_DIR / "results", OPT_RESULTS_DIR, OPT_DIR / "config",
                    OPT_DIR / "logs", EXISTING_DIR / "config"]
snapshot_before = {}
for folder in SNAPSHOT_FOLDERS:
    if folder.exists():
        snapshot_before.update(fingerprint_folder(folder))
training.save_json(MANIFEST_DIR / "snapshot_before.json", snapshot_before)
print(f"{len(snapshot_before)} existing result files fingerprinted.")

111 existing result files fingerprinted.


## 2. Final benchmark protocol and the frozen plan

| Item | Final benchmark |
|---|---|
| Dataset | `artifacts/events_prepared.parquet` (fingerprint in `artifacts/dataset_hash.json`) |
| Splits | match-level, 280 / 35 / 35 matches, as in Track A |
| Sequences | windows of K = 16 events, stride 8, never crossing a period |
| Target | 15 action classes |
| Evaluation | each event scored once, in the window where it has the most history before it |
| Metrics | NLL (primary), accuracy, top-3, top-5, macro-F1, ECE (15 bins) |
| Uncertainty | match-level bootstrap of per-event NLL averaged over seeds (2,000 resamples); seed SD reported separately |
| Seeds | multi-seed models: final seeds 100–104; deterministic baselines: one fit |
| Test policy | re-run baselines evaluated on test once, after their validation results are written; the selected Transformer is not re-evaluated |



In [ ]:
MLP_LEARNING_RATE = load_json(EXISTING_RESULTS_DIR / "04_b4_comparison.json")["selected_mlp_lr"]
ALPHAS = load_json(PROJECT_DIR / "results" / "baselines_validation.json")["selected_alpha"]

plan_path = CONFIG_DIR / "frozen_config.json"
plan = {
    "purpose": "Baselines B1a, B1b, B2, B3, B4 under the final benchmark conditions of the selected model",
    "selected_config_file": str(OPT_CONFIG_PATH),
    "selected_config_sha256": file_sha256(OPT_CONFIG_PATH),
    "encoding": ENCODING, "possession_features": POSSESSION_FEATURES,
    "context_length": FINAL_CONTEXT, "stride": STRIDE,
    "baselines": {
        "B1a class prior": {"fitted_on": "train", "hyperparameters": "none"},
        "B1b zone prior": {"alpha": ALPHAS["B1b zone prior"], "alpha_selected_in": "notebook 02 (validation)"},
        "B2 Markov": {"alpha": ALPHAS["B2 Markov"], "alpha_selected_in": "notebook 02 (validation)"},
        "B3 logistic regression": {"C": 1.0, "max_iter": 1000, "fixed_in": "notebook 02 (before training)"},
        "B4 MLP": {"learning_rate": MLP_LEARNING_RATE, "learning_rate_selected_in": "notebook 04 (validation)",
                   "weight_decay": SELECTED["training"]["weight_decay"],
                   "weight_decay_rule": "notebook 04: B4 uses the weight decay selected for the Transformer",
                   "max_epochs": SELECTED["training"]["max_epochs"], "seeds": FINAL_SEEDS,
                   "n_previous": 3, "hidden": 512},
    },
    "test_policy": "baselines evaluated once on test after the validation record is written; selected model not re-evaluated",
    "run_mode": RUN_MODE,
    "written_at": pd.Timestamp.now(tz="UTC").isoformat(),
}
if plan_path.exists():
    previous = load_json(plan_path)
    changed = {key for key in plan if key != "written_at" and previous.get(key) != plan[key]}
    if changed:
        raise RuntimeError(f"BLOCKED: the frozen plan already exists and differs in {sorted(changed)}.")
    print("Frozen plan already written on", previous["written_at"], "- unchanged.")
else:
    training.save_json(plan_path, plan)
    print("Frozen plan written before any fitting:", plan_path)
print(json.dumps(plan["baselines"], indent=1))

Frozen plan written before any fitting: /content/drive/MyDrive/soccer_mi/track_a/experiments/hyperparameter_optimisation/final_benchmark/config/frozen_config.json
{
 "B1a class prior": {
  "fitted_on": "train",
  "hyperparameters": "none"
 },
 "B1b zone prior": {
  "alpha": 10.0,
  "alpha_selected_in": "notebook 02 (validation)"
 },
 "B2 Markov": {
  "alpha": 10.0,
  "alpha_selected_in": "notebook 02 (validation)"
 },
 "B3 logistic regression": {
  "C": 1.0,
  "max_iter": 1000,
  "fixed_in": "notebook 02 (before training)"
 },
 "B4 MLP": {
  "learning_rate": 0.0009,
  "learning_rate_selected_in": "notebook 04 (validation)",
  "weight_decay": 0.01,
  "weight_decay_rule": "notebook 04: B4 uses the weight decay selected for the Transformer",
  "max_epochs": 30,
  "seeds": [
   100,
   101,
   102,
   103,
   104
  ],
  "n_previous": 3,
  "hidden": 512
 }
}


## 3. The selected model's own files



In [ ]:
final_table = pd.read_csv(OPT_RESULTS_DIR / "13_final_runs.csv")
final_validation = load_json(OPT_RESULTS_DIR / "13_final_validation.json")
sel_runs = [Path(path) for path in final_table.sort_values("seed")["run_dir"]]
PRIMARY_RUN_NAME = final_validation["primary_run"]
PRIMARY_SEED = final_validation["primary_seed"]

mismatches = []
for run_dir in sel_runs:
    run_config = load_json(run_dir / "run_config.json")
    for key in ["n_layers", "n_heads", "d_model", "d_ff"]:
        if run_config["model"][key] != SELECTED["model"][key]:
            mismatches.append((run_dir.name, key))
    if run_config["context_length"] != FINAL_CONTEXT:
        mismatches.append((run_dir.name, "context_length"))
    for key in ["learning_rate", "weight_decay"]:
        if abs(run_config["training"][key] - SELECTED["training"][key]) > 1e-12:
            mismatches.append((run_dir.name, key))
    if run_config.get("encoding") != ENCODING:
        mismatches.append((run_dir.name, "encoding"))
if mismatches:
    raise RuntimeError(f"BLOCKED: notebook 13 runs do not match the notebook 12 configuration: {mismatches}")
print(f"{len(sel_runs)} runs of the selected model found; all match the notebook 12 configuration.")
print(f"Primary checkpoint: seed {PRIMARY_SEED} ({PRIMARY_RUN_NAME})")
sel_summary = analysis.seed_summary(sel_runs)
sel_summary.round(4)

5 runs of the selected model found; all match the notebook 12 configuration.
Primary checkpoint: seed 103 (transformer_T300_pre_action_L4_H2_D128_F512_K16_g-all_lr0.0003_wd0.01_s103)


,run,seed,best epoch,epochs run,diverged,val NLL,val accuracy,val macro-F1,train NLL,minutes
0,transformer_T300_pre_action_L4_H2_D128_F512_K16_g-all_lr0.0003_wd0.01_s100,100,7,12,False,0.9067,0.6207,0.5456,0.8495,1.6084
1,transformer_T300_pre_action_L4_H2_D128_F512_K16_g-all_lr0.0003_wd0.01_s101,101,7,12,False,0.9018,0.6224,0.5656,0.8467,1.5911
2,transformer_T300_pre_action_L4_H2_D128_F512_K16_g-all_lr0.0003_wd0.01_s102,102,9,14,False,0.9056,0.6180,0.5601,0.8225,1.9635
3,transformer_T300_pre_action_L4_H2_D128_F512_K16_g-all_lr0.0003_wd0.01_s103,103,7,12,False,0.9044,0.6209,0.5575,0.8480,1.6829
4,transformer_T300_pre_action_L4_H2_D128_F512_K16_g-all_lr0.0003_wd0.01_s104,104,10,15,False,0.8982,0.6259,0.5630,0.8088,2.0174


## 4. Correctness tests at the selected configuration

### 4.1 Where the tests are and which depend on the configuration

The 16 tests are in `tests/test_model.py`. They were written and first executed on the development configuration (4 layers, 4 heads, K = 32, T100 encoding). Every test receives a `make_model` function and, except test 15, a `make_batch` function, so passing a model and windows built at the selected configuration re-executes the tests there without changing the test code.

The table records, before running, which tests were expected to depend on the configuration.

| # | Test | Depends on the model configuration? |
|---|---|---|
| 1 | shapes of every intermediate tensor | yes: expected shapes are computed from the model's own configuration |
| 2 | causal mask | yes: runs the model |
| 3 | attention equals PyTorch's reference | yes: per-head computation; head dimension changes from 32 to 64 |
| 4 | attention probabilities | yes |
| 5 | residual additions | yes: per block |
| 6 | LayerNorm placement | yes: per block |
| 7 | positional embeddings | yes: position table has K = 16 rows |
| 8 | output dimension | yes |
| 9 | determinism | yes |
| 10 | gradient flow | yes |
| 11 | loss and initial entropy | yes |
| 12 | masked positions do not affect the loss | yes |
| 13 | memorise one batch | yes |
| 14 | shuffled labels cannot beat the class prior | yes: trained briefly at the selected configuration |
| 15 | target-side columns cannot change the inputs | no: tests the feature-building code only; re-run with the T300 encoding |
| 16 | index-0 identity rows stay zero | yes: uses AdamW with weight decay 0.1 internally; the property tested (index-0 rows receive no update) does not depend on the weight-decay value |



### 4.2 The tested model is the selected model

Before the tests run, the parameter count of the test model is compared with the count recorded for the five final runs, and the test model's settings are taken from the frozen configuration.

**RESULT.** Test-model settings: 4 layers, 2 heads, d_model 128, d_ff 512, K = 16, T300. Parameters 911,119, equal to the recorded count.

In [ ]:
(PROJECT_DIR / "tests" / "__init__.py").touch()
for name in list(sys.modules):
    if name == "tests" or name.startswith("tests."):
        del sys.modules[name]
importlib.invalidate_caches()
import tests.test_model as model_tests
importlib.reload(model_tests)

check_encoding = data.load_encoding(ARTIFACTS_DIR / "encodings" / ENCODING)
check_features, check_vocabulary_sizes = data.encode_features(events, check_encoding, possession_features=POSSESSION_FEATURES)
check_encoded = batching.EncodedData(check_features, targets, DEVICE)
check_val_windows = data.build_windows(events, FINAL_CONTEXT, STRIDE, split="val")
check_train_windows = data.build_windows(events, FINAL_CONTEXT, STRIDE, split="train")
SELECTED_MODEL_SETTINGS = {"n_layers": SELECTED["model"]["n_layers"], "n_heads": SELECTED["model"]["n_heads"],
                           "d_model": SELECTED["model"]["d_model"], "d_ff": SELECTED["model"]["d_ff"],
                           "context_length": FINAL_CONTEXT}


def make_batch(n_windows=8, first_window=0):
    rows = torch.as_tensor(check_val_windows[first_window:first_window + n_windows], device=DEVICE)
    return check_encoded.batch(rows)


def make_model(seed=0, **overrides):
    torch.manual_seed(seed)
    settings = dict(SELECTED_MODEL_SETTINGS)
    settings.update(overrides)
    config = model_module.ModelConfig(vocabulary_sizes=check_vocabulary_sizes, **settings)
    return model_module.EventTransformer(config).to(DEVICE)


recorded_parameters = int(final_table["parameters"].iloc[0])
tested_parameters = model_module.count_parameters(make_model(0))["total"]
print("Test-model settings:", SELECTED_MODEL_SETTINGS, "| encoding", ENCODING)
print(f"Parameters of the test model: {tested_parameters:,} | recorded for the notebook 13 runs: {recorded_parameters:,}")
if tested_parameters != recorded_parameters:
    raise RuntimeError("BLOCKED: the test model is not the selected configuration.")

Test-model settings: {'n_layers': 4, 'n_heads': 2, 'd_model': 128, 'd_ff': 512, 'context_length': 16} | encoding T300
Parameters of the test model: 911,119 | recorded for the notebook 13 runs: 911,119


In [ ]:
previous_determinism = torch.are_deterministic_algorithms_enabled()
torch.use_deterministic_algorithms(True, warn_only=True)

generator = np.random.default_rng(0)
subset_windows = check_train_windows[generator.choice(len(check_train_windows), size=2000, replace=False)]
subset_rows = torch.as_tensor(subset_windows, device=DEVICE)
subset_inputs, subset_targets = check_encoded.batch(subset_rows)
permutation = torch.as_tensor(generator.permutation(subset_targets.numel()), device=DEVICE)
shuffled_targets = subset_targets.reshape(-1)[permutation].reshape(subset_targets.shape)
shuffled_label_batches = []
for epoch in range(2):
    for start in range(0, 2000, 64):
        batch_inputs = {name: tensor[start:start + 64] for name, tensor in subset_inputs.items()}
        shuffled_label_batches.append((batch_inputs, shuffled_targets[start:start + 64]))

validation_inputs, validation_targets = make_batch(400, 0)
train_class_counts = np.bincount(targets[events["split"].to_numpy() == "train"], minlength=15)
train_prior = train_class_counts / train_class_counts.sum()
prior_nll = float(-np.log(train_prior[validation_targets.cpu().numpy()]).mean())

test_context = {
    "make_model": make_model,
    "make_batch": make_batch,
    "shuffled_label_batches": shuffled_label_batches,
    "validation_batch": (validation_inputs, validation_targets),
    "prior_nll_on_validation_batch": prior_nll,
    "shuffled_label_tolerance": 0.01,
    "events": events,
    "encoding": check_encoding,
    "encode_features": data.encode_features,
}

start_time = time.time()
final_config_results = model_tests.run_all(test_context)
elapsed = time.time() - start_time
torch.use_deterministic_algorithms(previous_determinism)

development_results = load_json(PROJECT_DIR / "results" / "03_test_results.json")["results"]
development_passed = {row["test"]: row["passed"] for row in development_results}
purposes = {
    "1 shapes": "tensor shapes", "2 causal mask": "no access to later events",
    "3 attention vs reference": "equals PyTorch attention", "4 attention probabilities": "valid attention weights",
    "5 residual additions": "residual stream arithmetic", "6 LayerNorm placement": "pre-LN placement",
    "7 positional embeddings": "order information used", "8 output dimension": "15 classes",
    "9 determinism": "reproducibility", "10 gradient flow": "all parts trainable; index 0 frozen",
    "11 loss": "loss formula and initial entropy", "12 valid-position mask": "padding excluded from loss",
    "13 memorise one batch": "model can learn", "14 shuffled labels": "no target leakage",
    "15 target-side columns": "features independent of target", "16 index 0 stays zero": "index 0 has no meaning",
}
rows = []
for row in final_config_results:
    development = "PASS" if development_passed.get(row["test"]) else ("FAIL" if row["test"] in development_passed else "not recorded")
    final = "PASS" if row["passed"] else "FAIL"
    rows.append({"test": row["test"], "purpose": purposes.get(row["test"], ""),
                 "development config (4L_4H_K32, T100)": development,
                 "final config (4L_2H_K16, T300)": final,
                 "status": "PASS" if (final == "PASS" and development == "PASS") else "CHECK",
                 "detail (final config)": row["detail"]})
correctness_table = pd.DataFrame(rows).set_index("test")
correctness_table.to_csv(RESULTS_DIR / "correctness_tests_final_configuration.csv")
training.save_json(RESULTS_DIR / "15_correctness_final_config.json",
                   {"device": DEVICE, "torch": torch.__version__, "configuration": SELECTED_MODEL_SETTINGS,
                    "encoding": ENCODING, "tested_parameters": tested_parameters,
                    "test_module_sha256": file_sha256(PROJECT_DIR / "tests" / "test_model.py"),
                    "results": final_config_results, "seconds": elapsed})
n_passed = sum(row["passed"] for row in final_config_results)
print(f"{n_passed} of {len(final_config_results)} tests passed at the selected configuration in {elapsed:.0f} s on {DEVICE}")
correctness_table

14 of 16 tests passed at the selected configuration in 5 s on cuda


,purpose,"development config (4L_4H_K32, T100)","final config (4L_2H_K16, T300)",status,detail (final config)
test,,,,,
1 shapes,tensor shapes,PASS,PASS,PASS,"observed {'logits': (8, 16, 15), 'attention_probabilities': (8, 2, 16, 16), 'resid_post': (8, 16, 128)}"
2 causal mask,no access to later events,PASS,PASS,PASS,max change at positions <= 10: 0.0; after: 0.637
3 attention vs reference,equals PyTorch attention,PASS,FAIL,CHECK,ERROR: RuntimeError('The size of tensor a (16) must match the size of tensor b (32) at non-singleton dimension 3')
4 attention probabilities,valid attention weights,PASS,FAIL,CHECK,"ERROR: IndexError('The shape of the mask [32, 32] at index 0 does not match the shape of the indexed tensor [8, 2, 1..."
5 residual additions,residual stream arithmetic,PASS,PASS,PASS,max reconstruction error = 0.00e+00
6 LayerNorm placement,pre-LN placement,PASS,PASS,PASS,sub-layer inputs equal LN(residual): error 0.0e+00; explicit formula (x-mean)/sqrt(var+eps)*scale+shift: error 4.8e-07
7 positional embeddings,order information used,PASS,PASS,PASS,output change when positions removed 0.166; position gradient norm 0.248
8 output dimension,15 classes,PASS,PASS,PASS,classes = 15 (15 actions; END is not a target)
9 determinism,reproducibility,PASS,PASS,PASS,params identical True; outputs identical True; 10-step losses identical True


**How to read the table.** A cell says PASS only if the test was executed in that configuration and passed. The development column is read from the saved record of the first execution (`results/03_test_results.json`); the final column comes from the run above. `status` is PASS only if both are PASS.

**RESULT.** 14 of 16 tests pass at the selected configuration. **Tests 3 (attention against PyTorch's reference) and 4 (attention probabilities) fail with shape errors**: test 3 reports a size mismatch between 16 and 32, test 4 a mask of shape [32, 32] applied to 16-position attention. Their status is CHECK.

**INTERPRETATION.** The errors point to the test code, not the model: both tests build inputs or masks with a hard-coded sequence length of 32 (the development window), while the selected model's window is 16. The pre-run expectation in Section 4.1 (test 3 depends on the head dimension) did not anticipate this. The other 14 tests, including the causal-mask, residual, LayerNorm, determinism, shuffled-label and index-0 tests, pass with details consistent with the development run (for example, shuffled-label validation NLL 1.8580 against a class-prior NLL of 1.8537).



## 5. Baselines under the final conditions

### 5.1 B1a, B1b, B2, B3: refit from scratch

The refit helper is given a B3 checkpoint path inside this notebook's folder, which does not exist yet, so B3 is fitted from scratch rather than loaded. Validation predictions are computed now; test predictions only after the unlock in Section 8.

**RESULT.** B1–B3 fitted from scratch in 251 s. scikit-learn prints L-BFGS convergence warnings during the fit, but the fit reports 418 iterations and converged = True.

In [ ]:
features_for_baselines, _ = data.encode_features(events, check_encoding, POSSESSION_FEATURES)
train_rows = np.where(events["split"] == "train")[0]
val_rows = np.where(events["split"] == "val")[0]
final_bundle = training.prepare_data(events, ARTIFACTS_DIR, ENCODING, POSSESSION_FEATURES, FINAL_CONTEXT, STRIDE,
                                     DEVICE, RUN_MODE)
if RUN_MODE == "smoke":
    val_rows = np.sort(final_bundle["views"]["val"]["event_rows"])
    train_rows = train_rows[:20000]

fresh_b3_path = WORK_DIR / "artifacts_b3" / f"{ENCODING}_{POSSESSION_FEATURES}_fresh.joblib"
fresh_b3_path.parent.mkdir(parents=True, exist_ok=True)
b3_fitted_here = not fresh_b3_path.exists()
start_time = time.time()
non_neural_predict, fitted_lr = evaluation.fit_non_neural_baselines(features_for_baselines, targets, train_rows, ALPHAS,
                                                                    max_iter=1000 if RUN_MODE == "full" else 50,
                                                                    checkpoint_path=fresh_b3_path)
print(f"B1-B3 {'fitted from scratch' if b3_fitted_here else 'reloaded from this notebook folder'} in "
      f"{time.time() - start_time:.0f} s | logistic regression iterations {fitted_lr['iterations']}, "
      f"converged {fitted_lr['converged']}")
non_neural_val_logits = {name: predict(val_rows) for name, predict in non_neural_predict.items()}

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

B1-B3 fitted from scratch in 251 s | logistic regression iterations 418, converged True


### 5.2 B4: trained with the final training setting and the final seeds

Same configuration format as the Track A B4 runs; only the weight decay (0.01, the final Transformer setting) and the seeds (100–104, the final seed policy) differ, as recorded in the plan. The notebook stops if the final setting changes batch size or warm-up, since the B4 configuration would then need extending.

**RESULT.** Validation NLL by seed: 0.9473, 0.9486, 0.9490, 0.9505, 0.9459. Best epochs 3–4 (8–9 epochs run); no run diverged.

In [ ]:
def b4_config(seed):
    return {"model_type": "mlp", "encoding": ENCODING, "possession_features": POSSESSION_FEATURES,
            "context_length": FINAL_CONTEXT, "stride": STRIDE, "seed": seed,
            "model": {"n_layers": 0, "n_heads": 0, "d_model": 128, "d_ff": 0, "hidden": 512, "n_previous": 3},
            "training": {"learning_rate": MLP_LEARNING_RATE, "weight_decay": SELECTED["training"]["weight_decay"],
                         "max_epochs": SELECTED["training"]["max_epochs"]}}


for key in ["batch_size", "warmup_fraction"]:
    if SELECTED["training"][key] != training.DEFAULT_TRAINING[key]:
        raise RuntimeError(f"BLOCKED: the final training setting changes {key}; extend b4_config before training.")

b4_runs, b4_rows = [], []
for seed in FINAL_SEEDS:
    config = b4_config(seed)
    result = training.train_run(config, final_bundle, RUNS_DIR, MANIFEST_PATH, keep_every_epoch=False, verbose=True)
    run_dir = RUNS_DIR / result["run_name"]
    b4_runs.append(run_dir)
    b4_rows.append({"model": "B4 MLP", "seed": seed, "learning rate": MLP_LEARNING_RATE,
                    "weight decay": SELECTED["training"]["weight_decay"],
                    "val_nll": result["val_metrics"]["nll"], "val_accuracy": result["val_metrics"]["accuracy"],
                    "val_macro_f1": result["val_metrics"]["macro_f1"], "best_epoch": result["best_epoch"],
                    "epochs_run": result["epochs_run"], "diverged": result["diverged"], "run_dir": str(run_dir)})
b4_table = pd.DataFrame(b4_rows)
b4_table.to_csv(RESULTS_DIR / "15_b4_runs.csv", index=False)
b4_table.drop(columns=["run_dir"]).round(4)

[mlp_T300_pre_action_L0_H0_D128_F0_K16_g-all_lr0.0009_wd0.01_s100] epoch  1 | train loss 1.4718 | val NLL 1.1043 | val acc 0.5545 | val macro-F1 0.4693 | 3s
[mlp_T300_pre_action_L0_H0_D128_F0_K16_g-all_lr0.0009_wd0.01_s100] epoch  2 | train loss 1.0325 | val NLL 0.9813 | val acc 0.5954 | val macro-F1 0.5103 | 3s
[mlp_T300_pre_action_L0_H0_D128_F0_K16_g-all_lr0.0009_wd0.01_s100] epoch  3 | train loss 0.9258 | val NLL 0.9473 | val acc 0.6036 | val macro-F1 0.5408 | 3s
[mlp_T300_pre_action_L0_H0_D128_F0_K16_g-all_lr0.0009_wd0.01_s100] epoch  4 | train loss 0.8763 | val NLL 0.9492 | val acc 0.6048 | val macro-F1 0.5427 | 3s
[mlp_T300_pre_action_L0_H0_D128_F0_K16_g-all_lr0.0009_wd0.01_s100] epoch  5 | train loss 0.8355 | val NLL 0.9697 | val acc 0.5986 | val macro-F1 0.5414 | 3s
[mlp_T300_pre_action_L0_H0_D128_F0_K16_g-all_lr0.0009_wd0.01_s100] epoch  6 | train loss 0.7929 | val NLL 0.9944 | val acc 0.5929 | val macro-F1 0.5452 | 3s
[mlp_T300_pre_action_L0_H0_D128_F0_K16_g-all_lr0.0009_wd0.

,model,seed,learning rate,weight decay,val_nll,val_accuracy,val_macro_f1,best_epoch,epochs_run,diverged
0,B4 MLP,100,0.0009,0.01,0.9473,0.6036,0.5408,3,8,False
1,B4 MLP,101,0.0009,0.01,0.9486,0.6046,0.5468,4,9,False
2,B4 MLP,102,0.0009,0.01,0.9490,0.6052,0.5439,3,8,False
3,B4 MLP,103,0.0009,0.01,0.9505,0.5986,0.5437,4,9,False
4,B4 MLP,104,0.0009,0.01,0.9459,0.6069,0.5384,4,9,False


### 5.3 Validation results of every baseline

**RESULT.** Validation NLL: B1a 1.9746, B1b 1.7818, B2 1.4109, B3 1.0507, B4 0.9483 (seed SD 0.0018, 5 seeds). Accuracy rises from 0.3208 (B1a) to 0.6038 (B4), macro-F1 from 0.0324 to 0.5427. ECE is not computed for B4 in this table.

In [ ]:
val_targets = targets[val_rows]
validation_rows = []
for name, logits in non_neural_val_logits.items():
    m = metrics.compute_metrics(logits, val_targets)
    validation_rows.append({"model": name, "seeds": 1, "val NLL": m["nll"], "val NLL seed SD": 0.0,
                            "val accuracy": m["accuracy"], "val macro-F1": m["macro_f1"], "val ECE": m["ece"]})
b4_val = b4_table["val_nll"]
validation_rows.append({"model": "B4 MLP", "seeds": len(b4_runs), "val NLL": float(b4_val.mean()),
                        "val NLL seed SD": float(b4_val.std(ddof=1)), "val accuracy": float(b4_table["val_accuracy"].mean()),
                        "val macro-F1": float(b4_table["val_macro_f1"].mean()), "val ECE": np.nan})
baseline_validation_table = pd.DataFrame(validation_rows).set_index("model")
baseline_validation_table.to_csv(RESULTS_DIR / "15_baseline_validation.csv")
baseline_validation_table.round(4)

,seeds,val NLL,val NLL seed SD,val accuracy,val macro-F1,val ECE
model,,,,,,
B1a class prior,1,1.9746,0.0000,0.3208,0.0324,0.0185
B1b zone prior,1,1.7818,0.0000,0.3419,0.1353,0.0140
B2 Markov,1,1.4109,0.0000,0.4537,0.3588,0.0118
B3 logistic regression,1,1.0507,0.0000,0.5796,0.5073,0.0118
B4 MLP,5,0.9483,0.0018,0.6038,0.5427,NaN


## 6. Validation gates of the selected model against the re-run baselines

| Gate | Requirement | Origin |
|---|---|---|
| Baseline improvement | the selected model beats every one of B1–B4 beyond noise on validation | the decision rule |
| No divergence | no final seed diverged | objective |
| Stability | seed SD of validation NLL reported; no threshold | reported only |
| Class coverage | every class with more than 500 validation events has F1 > 0 in every seed | Track A convention (**PROVISIONAL**) |
| Size cap | at most 4 layers and 8 heads | approved plan |

**Decision rule ("better beyond noise").** Against B4 (five seeds): the 95% match-level bootstrap interval of the per-event NLL difference, computed on per-event NLL averaged over seeds, lies entirely below zero **and** every seed of the selected model has lower validation NLL than every seed of B4. The deterministic baselines B1a–B3 have one fit each; the selected model's seed-averaged per-event NLL is compared with that single fit (`evaluation.compare_runs_to_fixed`).

**RESULT.** Selected-model validation NLL over the five final seeds: mean 0.9033, SD 0.0034 (reported, no threshold). All four gates pass. Selected − baseline: B1a −1.0712, B1b −0.8785, B2 −0.5075, B3 −0.1474, B4 −0.0449 [−0.0479, −0.0418]; every comparison is beyond noise.

In [ ]:
val_rows_sorted, b8_val_nll = analysis.condition_event_nll(sel_runs, targets)
val_match_ids = match_ids[val_rows_sorted]
gate_rows = []
for name, logits in non_neural_val_logits.items():
    nll_fixed = metrics.per_event_nll(logits, val_targets)
    fixed_sorted = nll_fixed[np.argsort(val_rows)][np.searchsorted(np.sort(val_rows), val_rows_sorted)]
    comparison = evaluation.compare_runs_to_fixed(b8_val_nll, fixed_sorted, val_match_ids, n_resamples=N_BOOT)
    gate_rows.append({"baseline": name, "selected - baseline NLL": comparison["difference"],
                      "CI": f"[{comparison['ci_low']:+.4f}, {comparison['ci_high']:+.4f}]",
                      "better beyond noise": comparison["better_beyond_noise"]})
b4_comparison = analysis.compare_conditions(sel_runs, b4_runs, targets, match_ids)
gate_rows.append({"baseline": "B4 MLP", "selected - baseline NLL": b4_comparison["difference_a_minus_b"],
                  "CI": f"[{b4_comparison['difference_ci_low']:+.4f}, {b4_comparison['difference_ci_high']:+.4f}]",
                  "better beyond noise": b4_comparison["a_better_beyond_noise"]})
baseline_gate_table = pd.DataFrame(gate_rows).set_index("baseline")

val_support = np.bincount(targets[val_rows_sorted], minlength=15)
coverage_failures = []
for run_dir in sel_runs:
    per_class = analysis.load_final_results(run_dir)["val_metrics"]["per_class_f1"]
    for class_id in range(15):
        if val_support[class_id] > 500 and per_class[class_id] == 0:
            coverage_failures.append((run_dir.name, data.ACTION_NAMES[class_id]))

gates = {
    "baseline improvement (all of B1-B4)": bool(baseline_gate_table["better beyond noise"].all()),
    "no divergence": bool(not sel_summary["diverged"].any()),
    "class coverage (PROVISIONAL)": len(coverage_failures) == 0,
    "size cap": SELECTED["model"]["n_layers"] <= 4 and SELECTED["model"]["n_heads"] <= 8,
}
print("Validation NLL across the 5 final seeds: mean", round(sel_summary["val NLL"].mean(), 4),
      "| SD", round(sel_summary["val NLL"].std(ddof=1), 4), "(reported, no threshold)")
for gate, passed in gates.items():
    print(f"{'PASS' if passed else 'FAIL'}  {gate}")
if coverage_failures:
    print("Classes with zero F1:", coverage_failures)
baseline_gate_table

Validation NLL across the 5 final seeds: mean 0.9033 | SD 0.0034 (reported, no threshold)
PASS  baseline improvement (all of B1-B4)
PASS  no divergence
PASS  class coverage (PROVISIONAL)
PASS  size cap


,selected - baseline NLL,CI,better beyond noise
baseline,,,
B1a class prior,-1.071238,"[-1.0981, -1.0448]",True
B1b zone prior,-0.878518,"[-0.9000, -0.8562]",True
B2 Markov,-0.507523,"[-0.5210, -0.4926]",True
B3 logistic regression,-0.147388,"[-0.1542, -0.1400]",True
B4 MLP,-0.044931,"[-0.0479, -0.0418]",True


## 7. Every validation-based statement, recorded before the test split is unlocked



In [ ]:
validation_report = {
    "plan": plan,
    "selected_model_runs": [str(run) for run in sel_runs],
    "b4_runs": [str(run) for run in b4_runs],
    "b3_fitted_from_scratch_in_this_notebook": bool(b3_fitted_here),
    "b3_iterations": fitted_lr["iterations"], "b3_converged": fitted_lr["converged"],
    "baseline_validation": baseline_validation_table.reset_index().to_dict(orient="records"),
    "gates": gates,
    "baseline_comparison": baseline_gate_table.reset_index().to_dict(orient="records"),
    "coverage_failures": coverage_failures,
    "correctness_tests_passed": int(n_passed),
    "written_before_test_unlock": True,
    "run_mode": RUN_MODE,
    "written_at": pd.Timestamp.now(tz="UTC").isoformat(),
}
training.save_json(RESULTS_DIR / "15_validation_report.json", validation_report)
print("Validation record written before the test split is unlocked.")

Validation record written before the test split is unlocked.


## 8. Test evaluation of the re-run baselines (once)


**RESULT.** Test split unlocked once (2026-09-28 13:48 UTC) for the re-run baselines; 65,009 test events scored.

In [ ]:
def load_prediction_folder(folder, wanted=None):
    """{model name: [(run, logits sorted by event row), ...]} and the sorted event rows."""
    by_model = {}
    rows_reference = None
    for file in sorted(Path(folder).glob("*.npz")):
        stored = np.load(file, allow_pickle=True)
        name = str(stored["model"])
        if wanted is not None and name not in wanted:
            continue
        order = np.argsort(stored["event_rows"])
        rows = stored["event_rows"][order]
        if rows_reference is None:
            rows_reference = rows
        if not np.array_equal(rows, rows_reference):
            raise RuntimeError(f"BLOCKED: {file} covers different test events.")
        by_model.setdefault(name, []).append((str(stored["run"]), stored["logits"][order]))
    return by_model, rows_reference


def ordered_logits(entries):
    """Logits in run-name order (run names end in the seed, so this is seed order)."""
    return [logits for _, logits in sorted(entries, key=lambda item: item[0])]

In [ ]:
test_prediction_dir = RESULTS_DIR / "test_predictions"
test_prediction_dir.mkdir(parents=True, exist_ok=True)
test_done_flag = RESULTS_DIR / "15_test_results.json"

if test_done_flag.exists() or any(test_prediction_dir.glob("*.npz")):
    print("Baselines were already evaluated on test; saved predictions are reloaded (no new evaluation).")
    baseline_predictions, test_rows = load_prediction_folder(test_prediction_dir)
else:
    training.unlock_test_split(final_bundle, WORK_DIR, events)
    baseline_predictions = {}
    test_rows = None
    for run_dir in b4_runs:
        model = training.load_trained_model(run_dir, final_bundle)
        logits, _, rows = training.predict_view(model, final_bundle, "test")
        order = np.argsort(rows)
        logits, rows = logits[order], rows[order]
        if test_rows is None:
            test_rows = rows
        if not np.array_equal(rows, test_rows):
            raise RuntimeError("BLOCKED: B4 seeds scored different test events.")
        np.savez_compressed(test_prediction_dir / f"{run_dir.name}.npz", logits=logits, event_rows=rows,
                            model="B4 MLP", run=run_dir.name)
        baseline_predictions.setdefault("B4 MLP", []).append((run_dir.name, logits))
    for name, predict in non_neural_predict.items():
        logits = predict(test_rows)
        np.savez_compressed(test_prediction_dir / f"{name.replace(' ', '_')}.npz", logits=logits,
                            event_rows=test_rows, model=name, run=name)
        baseline_predictions[name] = [(name, logits)]
    print("Test split unlocked for the re-run baselines. Access log:")
    for line in open(LOG_DIR / "test_access_log.jsonl"):
        print(" ", line.strip())
test_targets = targets[test_rows]
test_match_ids = match_ids[test_rows]
if len(np.unique(test_rows)) != len(test_rows):
    raise RuntimeError("BLOCKED: duplicated test events.")
print("Test events scored by the re-run baselines:", len(test_rows))

Test split unlocked for the re-run baselines. Access log:
  {"time": "2026-09-28T13:48:35.480534+00:00", "frozen_config": "/content/drive/MyDrive/soccer_mi/track_a/experiments/hyperparameter_optimisation/final_benchmark/config/frozen_config.json", "encoding": "T300"}
Test events scored by the re-run baselines: 65009


In [ ]:
def headline_row(name, seed_logits, **extra):
    """Headline test metrics in the notebook 06 / 13 format (mean over seeds, seed SD, match-bootstrap CI)."""
    seed_metrics = [metrics.compute_metrics(logits, test_targets) for logits in seed_logits]
    event_nll = np.mean([metrics.per_event_nll(logits, test_targets) for logits in seed_logits], axis=0)
    interval = metrics.match_bootstrap_mean(event_nll, test_match_ids, n_resamples=N_BOOT)
    return {"model": name, **extra, "seeds": len(seed_logits), "test events": int(len(test_targets)),
            "NLL": float(np.mean([m["nll"] for m in seed_metrics])),
            "NLL seed SD": float(np.std([m["nll"] for m in seed_metrics], ddof=1)) if len(seed_metrics) > 1 else 0.0,
            "NLL 95% CI": f"[{interval['ci_low']:.4f}, {interval['ci_high']:.4f}]",
            "accuracy": float(np.mean([m["accuracy"] for m in seed_metrics])),
            "top-3": float(np.mean([m["top3_accuracy"] for m in seed_metrics])),
            "top-5": float(np.mean([m["top5_accuracy"] for m in seed_metrics])),
            "macro-F1": float(np.mean([m["macro_f1"] for m in seed_metrics])),
            "ECE": float(np.mean([m["ece"] for m in seed_metrics]))}


def seed_mean_event_nll(seed_logits):
    return np.mean([metrics.per_event_nll(logits, test_targets) for logits in seed_logits], axis=0)


def paired_test_contrast(logits_a, logits_b, mask=None):
    """Mean of NLL(a) - NLL(b) on test with a match-bootstrap interval and the seed-separation check."""
    keep = np.ones(len(test_targets), dtype=bool) if mask is None else mask
    difference = seed_mean_event_nll(logits_a)[keep] - seed_mean_event_nll(logits_b)[keep]
    interval = metrics.match_bootstrap_mean(difference, test_match_ids[keep], n_resamples=N_BOOT)
    seeds_a = [metrics.per_event_nll(l, test_targets)[keep].mean() for l in logits_a]
    seeds_b = [metrics.per_event_nll(l, test_targets)[keep].mean() for l in logits_b]
    separated = bool(max(seeds_a) < min(seeds_b)) if interval["mean"] < 0 else bool(min(seeds_a) > max(seeds_b))
    return {"events": int(keep.sum()), "difference": interval["mean"], "ci_low": interval["ci_low"],
            "ci_high": interval["ci_high"], "seeds separated": separated,
            "beyond noise": bool(separated and (interval["ci_high"] < 0 or interval["ci_low"] > 0))}

### 8.1 Baseline results under the final conditions

Every number in this table is newly computed in this notebook. For B4 the NLL is the mean over five seeds with seed SD and a 95% match-level bootstrap interval; B1a–B3 are single deterministic fits (seed SD 0).

**RESULT.** Test NLL: B1a 2.0202, B1b 1.8188, B2 1.4332, B3 1.0649, B4 0.9559 (seed SD 0.0036) [0.9434, 0.9681]. Each added source of information lowers test NLL, as on validation.

In [ ]:
BASELINE_ORDER = ["B1a class prior", "B1b zone prior", "B2 Markov", "B3 logistic regression", "B4 MLP"]
PROTOCOL = {
    "B1a class prior": "training class frequencies; refit",
    "B1b zone prior": f"zone frequencies, alpha {ALPHAS['B1b zone prior']:g} (nb 02); refit",
    "B2 Markov": f"(previous action, zone) frequencies, alpha {ALPHAS['B2 Markov']:g} (nb 02); refit",
    "B3 logistic regression": "current-event inputs, C = 1.0, <= 1,000 L-BFGS iterations (nb 02); fitted from scratch",
    "B4 MLP": f"current + 3 previous events; lr {MLP_LEARNING_RATE:g} (nb 04), weight decay "
              f"{SELECTED['training']['weight_decay']:g} (final setting), seeds {FINAL_SEEDS}",
}
baseline_benchmark = pd.DataFrame([
    headline_row(name, ordered_logits(baseline_predictions[name]), **{"training protocol": PROTOCOL[name],
                 "number source": "newly computed (nb 15)"})
    for name in BASELINE_ORDER]).set_index("model")
baseline_benchmark.to_csv(RESULTS_DIR / "baseline_final_benchmark.csv")
baseline_benchmark.round(4)

,training protocol,number source,seeds,test events,NLL,NLL seed SD,NLL 95% CI,accuracy,top-3,top-5,macro-F1,ECE
model,,,,,,,,,,,,
B1a class prior,training class frequencies; refit,newly computed (nb 15),1,65009,2.0202,0.0000,"[1.9945, 2.0468]",0.2971,0.7163,0.8167,0.0305,0.0052
B1b zone prior,"zone frequencies, alpha 10 (nb 02); refit",newly computed (nb 15),1,65009,1.8188,0.0000,"[1.7946, 1.8444]",0.3227,0.7504,0.8702,0.1327,0.0061
B2 Markov,"(previous action, zone) frequencies, alpha 10 (nb 02); refit",newly computed (nb 15),1,65009,1.4332,0.0000,"[1.4108, 1.4564]",0.4489,0.8434,0.9459,0.3611,0.0048
B3 logistic regression,"current-event inputs, C = 1.0, <= 1,000 L-BFGS iterations (nb 02); fitted from scratch",newly computed (nb 15),1,65009,1.0649,0.0000,"[1.0451, 1.0843]",0.5753,0.9198,0.9830,0.5090,0.0122
B4 MLP,"current + 3 previous events; lr 0.0009 (nb 04), weight decay 0.01 (final setting), seeds [100, 101, 102, 103, 104]",newly computed (nb 15),5,65009,0.9559,0.0036,"[0.9434, 0.9681]",0.6023,0.9384,0.9918,0.5452,0.0244


### 8.2 What changed relative to the original baseline benchmark



In [ ]:
track_a_predictions, track_a_rows = load_prediction_folder(EXISTING_RESULTS_DIR / "test_predictions")
if not np.array_equal(track_a_rows, test_rows):
    raise RuntimeError("BLOCKED: the original baseline predictions cover different test events.")
change_rows = []
for name in BASELINE_ORDER:
    original = ordered_logits(track_a_predictions[name])
    rerun = ordered_logits(baseline_predictions[name])
    original_row = headline_row(name, original)
    rerun_row = baseline_benchmark.loc[name]
    row = {"model": name, "original seeds": len(original), "re-run seeds": len(rerun),
           "original test NLL": original_row["NLL"], "re-run test NLL": rerun_row["NLL"],
           "difference (re-run - original)": rerun_row["NLL"] - original_row["NLL"],
           "original macro-F1": original_row["macro-F1"], "re-run macro-F1": rerun_row["macro-F1"]}
    if len(original) == 1 and len(rerun) == 1:
        row["max |logit difference|"] = float(np.abs(original[0] - rerun[0]).max())
    change_rows.append(row)
change_table = pd.DataFrame(change_rows).set_index("model")
change_table.to_csv(RESULTS_DIR / "15_original_vs_rerun_baselines.csv")
change_table.round(6)

,original seeds,re-run seeds,original test NLL,re-run test NLL,difference (re-run - original),original macro-F1,re-run macro-F1,max |logit difference|
model,,,,,,,,
B1a class prior,1,1,2.020198,2.020198,0.000000,0.030540,0.030540,0.0
B1b zone prior,1,1,1.818808,1.818808,0.000000,0.132674,0.132674,0.0
B2 Markov,1,1,1.433183,1.433183,0.000000,0.361073,0.361073,0.0
B3 logistic regression,1,1,1.064879,1.064879,0.000000,0.508976,0.508976,0.0
B4 MLP,3,5,0.952387,0.955876,0.003489,0.541263,0.545247,NaN


**RESULT.** B1a, B1b, B2 and B3 reproduce the original test predictions exactly (largest logit difference 0.0; identical NLL and macro-F1). B4 under the final setup (weight decay 0.01, 5 seeds) has test NLL 0.955876, against 0.952387 originally (weight decay 0.1, 3 seeds): 0.003489 higher. Its macro-F1 is slightly higher (0.5452 against 0.5413).

**INTERPRETATION.** The earlier B1–B3 numbers were already valid under the final conditions. B4 is slightly worse in NLL under the final setup; the re-run B4 is the one used for the final comparison, which is the less favourable choice for the Transformer. Neither result is used for any decision.

## 9. Transformer results, loaded from the recorded files


In [ ]:
recorded_13 = pd.DataFrame(load_json(OPT_RESULTS_DIR / "13_test_results.json")["headline"]).set_index("model")
recorded_4l = pd.DataFrame(load_json(BENCH_DIR / "results" / "06_4L_test_results.json")["headline"]).set_index("model")
print("Imported from 13_test_results.json:")
display(recorded_13.round(4))

sel_predictions, sel_rows = load_prediction_folder(OPT_RESULTS_DIR / "test_predictions", wanted={"optimised model"})
ref4_predictions, ref4_rows = load_prediction_folder(BENCH_DIR / "results" / "test_predictions",
                                                    wanted={"B8 Transformer 4L (selected)"})
for label, rows in [("notebook 13", sel_rows), ("4-layer notebook 06", ref4_rows)]:
    if not np.array_equal(rows, test_rows):
        raise RuntimeError(f"BLOCKED: {label} predictions cover different test events.")
TRANSFORMER_SETS = {
    "REF-2L (Track A model)": (ordered_logits(track_a_predictions["B8 Transformer (final)"]),
                               recorded_13.loc["Track A model (frozen)"], "13_test_results.json (via nb 13/14); predictions from Track A nb 06"),
    "REF-4L (4-layer benchmark model)": (ordered_logits(ref4_predictions["B8 Transformer 4L (selected)"]),
                                         recorded_4l.loc["B8 Transformer 4L (selected)"], "06_4L_test_results.json; 4-layer nb 06"),
    "SEL-4L (selected model)": (ordered_logits(sel_predictions["optimised model"]),
                                recorded_13.loc["Optimised model"], "13_test_results.json (via nb 13/14); predictions from nb 13"),
}
for name, (seed_logits, recorded_row, source) in TRANSFORMER_SETS.items():
    recomputed = headline_row(name, seed_logits)
    for metric in ["NLL", "accuracy", "macro-F1", "ECE"]:
        if abs(recomputed[metric] - float(recorded_row[metric])) > 1e-6:
            raise RuntimeError(f"BLOCKED: {name} {metric} recomputed {recomputed[metric]} differs from recorded {recorded_row[metric]}.")
    print(f"{name}: recomputed NLL {recomputed['NLL']:.6f} = recorded {float(recorded_row['NLL']):.6f}")

Imported from 13_test_results.json:


,seeds,NLL,NLL seed SD,NLL 95% CI,accuracy,top-3,top-5,macro-F1,ECE
model,,,,,,,,,
Track A model (frozen),5,0.9158,0.0013,"[0.9035, 0.9283]",0.6161,0.9445,0.9937,0.5580,0.0238
Optimised model,5,0.9148,0.0032,"[0.9023, 0.9273]",0.6171,0.9440,0.9936,0.5585,0.0270


REF-2L (Track A model): recomputed NLL 0.915847 = recorded 0.915847
REF-4L (4-layer benchmark model): recomputed NLL 0.914042 = recorded 0.914042
SEL-4L (selected model): recomputed NLL 0.914801 = recorded 0.914801


## 10. Combined benchmark table

The columns follow the headline format used throughout the project: seeds, NLL (mean over seeds), NLL seed SD, 95% match-level bootstrap interval, accuracy, top-3, top-5, macro-F1 and ECE, each kept separate.

| Column | Meaning |
|---|---|
| family | baseline or Transformer |
| benchmark stage | where the row's model was trained and evaluated |
| training setup | the setting the row's model was trained under |
| number source | newly computed here, or imported from a recorded result file |


In [ ]:
combined_rows = []
for name in BASELINE_ORDER:
    row = baseline_benchmark.loc[name].to_dict()
    combined_rows.append({"model": name, "family": "baseline", "benchmark stage": "final benchmark (nb 15)",
                          "training setup": "final (weight decay 0.01, seeds 100-104)" if name == "B4 MLP" else "no training setting (fit on train)",
                          **{k: row[k] for k in ["seeds", "test events", "NLL", "NLL seed SD", "NLL 95% CI", "accuracy",
                                                 "top-3", "top-5", "macro-F1", "ECE"]},
                          "number source": "newly computed (nb 15)", "source file": "final_benchmark/results/test_predictions"})
STAGES = {"REF-2L (Track A model)": ("Phase 1 reference (Track A nb 06)", "Track A recipe (weight decay 0.1)"),
          "REF-4L (4-layer benchmark model)": ("Phase 1 depth check (4-layer nb 06)", "Track A recipe (weight decay 0.1)"),
          "SEL-4L (selected model)": ("final benchmark (nb 13; selected in nb 12)", "final (weight decay 0.01)")}
for name, (seed_logits, recorded_row, source) in TRANSFORMER_SETS.items():
    recomputed = headline_row(name, seed_logits)
    combined_rows.append({"model": name, "family": "Transformer", "benchmark stage": STAGES[name][0],
                          "training setup": STAGES[name][1],
                          "seeds": int(recorded_row["seeds"]), "test events": int(len(test_rows)),
                          "NLL": float(recorded_row["NLL"]), "NLL seed SD": float(recorded_row["NLL seed SD"]),
                          "NLL 95% CI": recorded_row["NLL 95% CI"], "accuracy": float(recorded_row["accuracy"]),
                          "top-3": float(recorded_row["top-3"]), "top-5": float(recorded_row["top-5"]),
                          "macro-F1": float(recorded_row["macro-F1"]), "ECE": float(recorded_row["ECE"]),
                          "number source": "imported from recorded result file (checked against saved predictions)",
                          "source file": source})
combined = pd.DataFrame(combined_rows).set_index("model")
combined.to_csv(RESULTS_DIR / "combined_final_benchmark.csv")
combined.round(4)

,family,benchmark stage,training setup,seeds,test events,NLL,NLL seed SD,NLL 95% CI,accuracy,top-3,top-5,macro-F1,ECE,number source,source file
model,,,,,,,,,,,,,,,
B1a class prior,baseline,final benchmark (nb 15),no training setting (fit on train),1,65009,2.0202,0.0000,"[1.9945, 2.0468]",0.2971,0.7163,0.8167,0.0305,0.0052,newly computed (nb 15),final_benchmark/results/test_predictions
B1b zone prior,baseline,final benchmark (nb 15),no training setting (fit on train),1,65009,1.8188,0.0000,"[1.7946, 1.8444]",0.3227,0.7504,0.8702,0.1327,0.0061,newly computed (nb 15),final_benchmark/results/test_predictions
B2 Markov,baseline,final benchmark (nb 15),no training setting (fit on train),1,65009,1.4332,0.0000,"[1.4108, 1.4564]",0.4489,0.8434,0.9459,0.3611,0.0048,newly computed (nb 15),final_benchmark/results/test_predictions
B3 logistic regression,baseline,final benchmark (nb 15),no training setting (fit on train),1,65009,1.0649,0.0000,"[1.0451, 1.0843]",0.5753,0.9198,0.9830,0.5090,0.0122,newly computed (nb 15),final_benchmark/results/test_predictions
B4 MLP,baseline,final benchmark (nb 15),"final (weight decay 0.01, seeds 100-104)",5,65009,0.9559,0.0036,"[0.9434, 0.9681]",0.6023,0.9384,0.9918,0.5452,0.0244,newly computed (nb 15),final_benchmark/results/test_predictions
REF-2L (Track A model),Transformer,Phase 1 reference (Track A nb 06),Track A recipe (weight decay 0.1),5,65009,0.9158,0.0013,"[0.9035, 0.9283]",0.6161,0.9445,0.9937,0.5580,0.0238,imported from recorded result file (checked against saved predictions),13_test_results.json (via nb 13/14); predictions from Track A nb 06
REF-4L (4-layer benchmark model),Transformer,Phase 1 depth check (4-layer nb 06),Track A recipe (weight decay 0.1),5,65009,0.9140,0.0033,"[0.9017, 0.9265]",0.6178,0.9442,0.9937,0.5578,0.0255,imported from recorded result file (checked against saved predictions),06_4L_test_results.json; 4-layer nb 06
SEL-4L (selected model),Transformer,final benchmark (nb 13; selected in nb 12),final (weight decay 0.01),5,65009,0.9148,0.0032,"[0.9023, 0.9273]",0.6171,0.9440,0.9936,0.5585,0.0270,imported from recorded result file (checked against saved predictions),13_test_results.json (via nb 13/14); predictions from nb 13


**RESULT (combined table).** Test NLL: B1a 2.0202, B1b 1.8188, B2 1.4332, B3 1.0649, B4 0.9559, REF-2L 0.9158, REF-4L 0.9140, SEL-4L 0.9148. The three Transformer rows lie between 0.9140 and 0.9158, with overlapping bootstrap intervals, and all are below B4.

### 10.1 The selected model against each re-run baseline on test (descriptive)

Paired per-event differences on the same test events, with match-bootstrap intervals and seed separation. These describe the final comparison; no decision depends on them. The same comparison on validation is the gate of Section 6.

In [ ]:
sel_logits = TRANSFORMER_SETS["SEL-4L (selected model)"][0]
versus_rows = []
for name in BASELINE_ORDER:
    contrast = paired_test_contrast(sel_logits, ordered_logits(baseline_predictions[name]))
    versus_rows.append({"baseline": name, "selected - baseline NLL": contrast["difference"],
                        "95% CI": f"[{contrast['ci_low']:+.4f}, {contrast['ci_high']:+.4f}]",
                        "seeds separated": contrast["seeds separated"], "beyond noise": contrast["beyond noise"]})
versus_table = pd.DataFrame(versus_rows).set_index("baseline")
versus_table.to_csv(RESULTS_DIR / "15_selected_vs_baselines_test.csv")
versus_table.round(4)

,selected - baseline NLL,95% CI,seeds separated,beyond noise
baseline,,,,
B1a class prior,-1.1054,"[-1.1270, -1.0840]",True,True
B1b zone prior,-0.9040,"[-0.9232, -0.8844]",True,True
B2 Markov,-0.5184,"[-0.5362, -0.5002]",True,True
B3 logistic regression,-0.1501,"[-0.1629, -0.1373]",True,True
B4 MLP,-0.0411,"[-0.0438, -0.0381]",True,True


**RESULT.** SEL-4L − baseline on test: B1a −1.1054, B1b −0.9040, B2 −0.5184, B3 −0.1501 [−0.1629, −0.1373], B4 −0.0411 [−0.0438, −0.0381]. Every interval lies below zero and the seeds are separated in every case: beyond noise against all five baselines, as on validation.

### 10.2 Additional statistics for the paper, from saved predictions only

No model is evaluated here.

1. The same architecture trained with the two training setups: REF-4L (weight decay 0.1) against SEL-4L (weight decay 0.01), on validation and on test. Both use seeds 100–104.
2. Reliability table of the selected model's primary checkpoint (seed 103) on test.
3. Per-class test F1 of the selected model, as the mean over the five final seeds.
4. Distribution of scoring positions in the evaluation views at K = 16.

In [ ]:
ref4_report = load_json(BENCH_DIR / "results" / "06_4L_validation_report.json")
ref4_runs = [BENCH_DIR / "runs" / row["run"] for row in sorted(ref4_report["seed_summary"], key=lambda r: r["seed"])]
phase_val = analysis.compare_conditions(sel_runs, ref4_runs, targets, match_ids)
phase_test = paired_test_contrast(sel_logits, TRANSFORMER_SETS["REF-4L (4-layer benchmark model)"][0])
per_seed = pd.DataFrame({"seed": FINAL_SEEDS,
                         "REF-4L val NLL (wd 0.1)": [analysis.load_final_results(r)["val_metrics"]["nll"] for r in ref4_runs],
                         "SEL-4L val NLL (wd 0.01)": [analysis.load_final_results(r)["val_metrics"]["nll"] for r in sel_runs]})
per_seed["difference"] = per_seed["SEL-4L val NLL (wd 0.01)"] - per_seed["REF-4L val NLL (wd 0.1)"]
print(f"SEL-4L - REF-4L, validation: {phase_val['difference_a_minus_b']:+.4f} "
      f"[{phase_val['difference_ci_low']:+.4f}, {phase_val['difference_ci_high']:+.4f}], beyond noise {phase_val['a_better_beyond_noise']}")
print(f"SEL-4L - REF-4L, test: {phase_test['difference']:+.4f} [{phase_test['ci_low']:+.4f}, {phase_test['ci_high']:+.4f}], "
      f"seeds separated {phase_test['seeds separated']}, beyond noise {phase_test['beyond noise']}")
display(per_seed.round(4))

sel_entries = sorted(sel_predictions["optimised model"], key=lambda item: item[0])
primary_logits = [logits for run, logits in sel_entries if run == PRIMARY_RUN_NAME][0]
reliability = pd.DataFrame(metrics.reliability_table(np.exp(metrics.log_softmax(primary_logits)), test_targets))
reliability.to_csv(RESULTS_DIR / "15_sel4l_primary_reliability_test.csv", index=False)
print(f"Reliability bins, primary checkpoint (seed {PRIMARY_SEED}), test:")
display(reliability.round(4))

per_class = np.mean([metrics.compute_metrics(l, test_targets)["per_class_f1"] for l in sel_logits], axis=0)
per_class_table = pd.DataFrame({"class": data.ACTION_NAMES, "test events": np.bincount(test_targets, minlength=15),
                                "F1 (mean over 5 final seeds)": per_class})
per_class_table.to_csv(RESULTS_DIR / "15_sel4l_per_class_f1_test.csv", index=False)
display(per_class_table.round(4))

position_rows = []
for split in ["val", "test"]:
    view = data.build_evaluation_view(events, split, FINAL_CONTEXT, STRIDE)
    counts = pd.Series(view["position"]).value_counts().sort_index()
    for position, count in counts.items():
        position_rows.append({"split": split, "position in window": int(position), "events": int(count),
                              "share": count / counts.sum()})
position_table = pd.DataFrame(position_rows)
position_table.to_csv(RESULTS_DIR / "15_scoring_positions_K16.csv", index=False)
position_table.pivot(index="position in window", columns="split", values="share").round(4)

SEL-4L - REF-4L, validation: +0.0003 [-0.0001, +0.0006], beyond noise False
SEL-4L - REF-4L, test: +0.0008 [+0.0004, +0.0011], seeds separated False, beyond noise False


,seed,REF-4L val NLL (wd 0.1),SEL-4L val NLL (wd 0.01),difference
0,100,0.9069,0.9067,-0.0003
1,101,0.9021,0.9018,-0.0004
2,102,0.9041,0.9056,0.0015
3,103,0.9038,0.9044,0.0006
4,104,0.8984,0.8982,-0.0002


Reliability bins, primary checkpoint (seed 103), test:


,bin_low,bin_high,count,mean_confidence,accuracy
0,0.0000,0.0667,0,NaN,NaN
1,0.0667,0.1333,0,NaN,NaN
2,0.1333,0.2000,0,NaN,NaN
3,0.2000,0.2667,123,0.2521,0.2602
4,0.2667,0.3333,1242,0.3090,0.3084
5,0.3333,0.4000,4115,0.3714,0.3628
6,0.4000,0.4667,7370,0.4356,0.4293
7,0.4667,0.5333,10677,0.5011,0.4876
8,0.5333,0.6000,9680,0.5659,0.5436
9,0.6000,0.6667,7653,0.6321,0.6059


,class,test events,F1 (mean over 5 final seeds)
0,PASS_SHORT,19314,0.5298
1,PASS_LONG,12973,0.5224
2,CROSS,878,0.3421
3,CARRY,14281,0.7688
4,DRIBBLE,1170,0.0417
5,SHOT,955,0.5841
6,CLEARANCE,1637,0.6043
7,RECOVERY,3861,0.6098
8,INTERCEPTION,2202,0.7320
9,DUEL,2667,0.9019


split,test,val
position in window,,
0,0.0011,0.0010
1,0.0011,0.0010
2,0.0011,0.0010
3,0.0011,0.0010
4,0.0011,0.0010
5,0.0011,0.0010
6,0.0011,0.0010
7,0.0011,0.0010
8,0.1235,0.1235


**RESULT.**

1. **REF-4L vs SEL-4L.** Validation: SEL-4L − REF-4L = +0.0003 [−0.0001, +0.0006], not beyond noise. Per seed the differences are between −0.0004 and +0.0015. Test: +0.0008 [+0.0004, +0.0011]; the interval excludes zero but the seeds are not separated, so under the decision rule the two are not distinguished. Changing the weight decay from 0.1 to 0.01 at 4 layers did not lower NLL; on test the point estimate is slightly higher with weight decay 0.01.
2. **Reliability (seed 103, test).** No events fall below confidence 0.20. In most bins observed accuracy is slightly below mean confidence (for example 0.5436 at 0.5659, 0.6546 at 0.6979), i.e. mild over-confidence in the middle range; the top bin (9,032 events) is close (0.9808 at 0.9835).
3. **Per-class F1.** High for FOUL_WON (0.9993), GK_ACTION (0.9387), DUEL (0.9019); near zero for DRIBBLE (0.0417), LOSS (0.0535) and OTHER (0.0762, 62 events).
4. **Scoring positions.** About 0.1% of events are scored at each of positions 0–7 and about 12.4% at each of positions 8–15, in both validation and test: almost every event is scored with at least 8 earlier events visible.

## 11. Sanity checks

In [ ]:
sanity = {}
sanity["all test sources cover the same events"] = True       # enforced when each source was loaded
sanity["no duplicated test events"] = len(np.unique(test_rows)) == len(test_rows)
sanity["metric code identical for every row (src/metrics.py, one import)"] = True
sanity["B4 has 5 seeds (final seed policy)"] = len(baseline_predictions["B4 MLP"]) == len(FINAL_SEEDS)
sanity["Transformer rows have 5 seeds"] = all(len(v[0]) == 5 for v in TRANSFORMER_SETS.values())
sanity["validation record written before the unlock"] = (RESULTS_DIR / "15_validation_report.json").exists()
access_log = LOG_DIR / "test_access_log.jsonl"
n_new_accesses = sum(1 for _ in open(access_log)) if access_log.exists() else 0
sanity["exactly one test access by this notebook"] = n_new_accesses == 1
sanity["selected model not retrained or re-evaluated"] = all(OPT_DIR in r.parents and WORK_DIR not in r.parents for r in sel_runs) \
    and not any(str(np.load(f, allow_pickle=True)["model"]) == "optimised model" for f in test_prediction_dir.glob("*.npz"))
sanity["all correctness tests pass at the final configuration"] = n_passed == len(final_config_results)
sanity["all notebook 06 gates pass against the re-run baselines"] = all(gates.values())
snapshot_after = {}
for folder in SNAPSHOT_FOLDERS:
    if folder.exists():
        snapshot_after.update(fingerprint_folder(folder))
changed = sorted(name for name, digest in snapshot_before.items() if snapshot_after.get(name) != digest)
sanity["no earlier result file modified"] = len(changed) == 0
for check, passed in sanity.items():
    print(f"{'PASS' if passed else 'FAIL'}  {check}")
if changed:
    print("Modified files:", changed)
training.save_json(RESULTS_DIR / "15_sanity_checks.json", {"checks": sanity, "modified_files": changed,
                                                          "new_test_accesses": n_new_accesses})

PASS  all test sources cover the same events
PASS  no duplicated test events
PASS  metric code identical for every row (src/metrics.py, one import)
PASS  B4 has 5 seeds (final seed policy)
PASS  Transformer rows have 5 seeds
PASS  validation record written before the unlock
PASS  exactly one test access by this notebook
PASS  selected model not retrained or re-evaluated
FAIL  all correctness tests pass at the final configuration
PASS  all notebook 06 gates pass against the re-run baselines
PASS  no earlier result file modified


**RESULT.** 10 of 11 sanity checks pass. **FAIL: "all correctness tests pass at the final configuration"** — this is the test 3 and test 4 failure of Section 4 (hard-coded sequence length 32). The remaining checks confirm: all test sources cover the same events with no duplicates; one metric implementation for every row; five seeds for B4 and for every Transformer row; the validation record was written before the unlock; exactly one test access by this notebook; the selected model was neither retrained nor re-evaluated; all gates pass; no earlier result file was modified.

## 12. Result record, manifest and summary

In [ ]:
test_record = {
    "run_mode": RUN_MODE, "n_test_events": int(len(test_rows)),
    "baseline_final_benchmark": baseline_benchmark.reset_index().to_dict(orient="records"),
    "original_vs_rerun_baselines": change_table.reset_index().to_dict(orient="records"),
    "combined_final_benchmark": combined.reset_index().to_dict(orient="records"),
    "selected_vs_baselines_test": versus_table.reset_index().to_dict(orient="records"),
    "sel4l_minus_ref4l_validation": phase_val, "sel4l_minus_ref4l_test": phase_test,
    "written_at": pd.Timestamp.now(tz="UTC").isoformat(),
}
if not test_done_flag.exists():
    training.save_json(test_done_flag, test_record)
    print("Test record saved:", test_done_flag)
else:
    print("Test record exists from the first run and is not overwritten.")

manifest = {
    "notebook": "15_final_benchmark",
    "baseline_models": {name: plan["baselines"][name] for name in BASELINE_ORDER},
    "final_transformer_configurations": {
        "SEL-4L": {"config_file": str(OPT_CONFIG_PATH), "sha256": plan["selected_config_sha256"],
                   "context_length": FINAL_CONTEXT, "model": SELECTED["model"], "training": SELECTED["training"]},
        "REF-2L": {"config_file": str(EXISTING_FROZEN_CONFIG)},
        "REF-4L": {"config_file": str(BENCH_DIR / "config" / "frozen_config.json")}},
    "seeds": {"B4": FINAL_SEEDS, "Transformers": FINAL_SEEDS, "B1a-B3": "deterministic, one fit"},
    "dataset": {"events": int(len(events)), "encoding": ENCODING, "possession_features": POSSESSION_FEATURES,
                "splits": "match-level 280/35/35", "validation events": int(len(val_rows_sorted)),
                "test events": int(len(test_rows))},
    "metrics": ["NLL (primary)", "accuracy", "top-3", "top-5", "macro-F1", "ECE (15 bins)"],
    "uncertainty": f"match-level bootstrap, {N_BOOT} resamples; seed SD reported separately",
    "source_notebooks": {"data": "Track A 01-02", "baseline definitions": "Track A 02 (B1-B3), 04 (B4)",
                         "tests": "Track A 03", "decision rule": "Track A 04", "gates and refit": "Track A 06",
                         "selected configuration": "12", "selected model results": "13 (loaded by 14)",
                         "REF-4L": "4-layer 06"},
    "result_files": sorted(str(p.relative_to(WORK_DIR)) for p in RESULTS_DIR.glob("*") if p.is_file()),
    "test_accesses_by_this_notebook": n_new_accesses,
    "environment": training.environment_info(),
}
training.save_json(MANIFEST_DIR / "15_experiment_manifest.json", manifest)
print("Manifest written:", MANIFEST_DIR / "15_experiment_manifest.json")

Test record saved: /content/drive/MyDrive/soccer_mi/track_a/experiments/hyperparameter_optimisation/final_benchmark/results/15_test_results.json
Manifest written: /content/drive/MyDrive/soccer_mi/track_a/experiments/hyperparameter_optimisation/final_benchmark/manifests/15_experiment_manifest.json


In [ ]:
b4_change = change_table.loc["B4 MLP"]
b13_max = max(change_table.loc[name].get("max |logit difference|", np.nan) for name in BASELINE_ORDER[:4])
sel_row = combined.loc["SEL-4L (selected model)"]
lines = [
    "WHAT WAS RE-RUN",
    f"- B1a, B1b, B2, B3 refit from scratch on the training split (B3 converged: {fitted_lr['converged']}).",
    f"- B4 trained with the final training setting (weight decay {SELECTED['training']['weight_decay']:g}) and final seeds {FINAL_SEEDS}.",
    "- All five baselines evaluated once on test, after their validation results were written.",
    f"- The 16 correctness tests at the selected configuration: {n_passed} of {len(final_config_results)} passed.",
    "WHAT WAS REUSED",
    "- Baseline definitions (nb 02, 04), smoothing strengths and the B4 learning rate, all selected on validation earlier; none re-tuned.",
    "- The selected model's runs and test results (nb 13, loaded by nb 14); REF-2L and REF-4L results.",
    "WHAT CHANGED FROM THE ORIGINAL BASELINE BENCHMARK",
    f"- B4: weight decay 0.1 -> {SELECTED['training']['weight_decay']:g}, seeds 0-2 -> {FINAL_SEEDS}; "
    f"test NLL {b4_change['original test NLL']:.4f} -> {b4_change['re-run test NLL']:.4f}.",
    f"- B1a-B3: no setting changed; largest logit difference from the original fits {b13_max:.2e}.",
    "WHAT REMAINED IDENTICAL",
    "- Data, preprocessing, splits, windows, evaluation view, metric code, bootstrap, decision rule, test events.",
    "WHY THE COMPARISON IS SCIENTIFICALLY VALID",
    f"- Every row is scored on the same {len(test_rows):,} test events with the same metric code, checked row by row.",
    "- The baselines use the same training setup as the selected model wherever a baseline has a training setup.",
    "- No test result was used for any decision; the baselines were frozen and their validation record written before the unlock.",
    "WHAT EVIDENCE WAS ADDED FOR CORRECTNESS ON THE FINAL CONFIGURATION",
    f"- The unchanged test module (sha256 recorded) executed at 4L_2H_K16, T300: {n_passed}/{len(final_config_results)} passed; "
    f"the tested model has the recorded parameter count ({tested_parameters:,}).",
    "RESULTS",
    "- Gates: " + ", ".join(f"{k}: {'pass' if v else 'FAIL'}" for k, v in gates.items()),
    f"- Selected model on test: NLL {sel_row['NLL']:.4f} {sel_row['NLL 95% CI']} (recorded in nb 13).",
    "WHAT THIS DOES NOT SHOW",
    "- Anything about the model's internal computation, or about data outside this study.",
]
print("\n".join(lines))
training.save_json(RESULTS_DIR / "15_summary.json", {"lines": lines})

WHAT WAS RE-RUN
- B1a, B1b, B2, B3 refit from scratch on the training split (B3 converged: True).
- B4 trained with the final training setting (weight decay 0.01) and final seeds [100, 101, 102, 103, 104].
- All five baselines evaluated once on test, after their validation results were written.
- The 16 correctness tests at the selected configuration: 14 of 16 passed.
WHAT WAS REUSED
- Baseline definitions (nb 02, 04), smoothing strengths and the B4 learning rate, all selected on validation earlier; none re-tuned.
- The selected model's runs and test results (nb 13, loaded by nb 14); REF-2L and REF-4L results.
WHAT CHANGED FROM THE ORIGINAL BASELINE BENCHMARK
- B4: weight decay 0.1 -> 0.01, seeds 0-2 -> [100, 101, 102, 103, 104]; test NLL 0.9524 -> 0.9559.
- B1a-B3: no setting changed; largest logit difference from the original fits 0.00e+00.
WHAT REMAINED IDENTICAL
- Data, preprocessing, splits, windows, evaluation view, metric code, bootstrap, decision rule, test events.
WHY THE COMP

## 13. Research conclusions

What this notebook establishes, from the outputs above:

* The baselines B1a–B4 have been trained and evaluated under the same final conditions as the selected Transformer, on the same 65,009 test events, with the same metric code. B1a–B3 reproduce the original benchmark exactly. B4 under the final setup has test NLL 0.9559 (originally 0.9524).
* The selected model passes every pre-registered validation gate against the re-run baselines, and on test it is below every baseline beyond noise (against B4: −0.0411 [−0.0438, −0.0381]).
* The combined table places the re-run baselines and the three recorded Transformer results side by side, with each row's source and training setup recorded. The Transformer rows (REF-2L 0.9158, REF-4L 0.9140, SEL-4L 0.9148) are not separated from one another by this table, and they were not trained under one procedure.
* 14 of the 16 correctness tests pass at the selected configuration as executed here. Tests 3 and 4 fail because the test code fixes the sequence length at 32.

**What may be stated in the paper.** "The baselines were re-evaluated under the final framework" is supported by these outputs. "The correctness tests were re-executed at the final configuration" is supported only together with the separate re-run of tests 3 and 4 with the sequence length read from the model (`15b_correctness_tests_3_4.json`); from this notebook alone, one sanity check fails and the failure must be reported rather than the statement.

**What this does not show.** Anything about the model's internal computation, or about data outside this study.